# Calculating phonons with **`exciting`** using DFPT
By: Sebastian Tillack

**<span style="color:firebrick">Note</span>**: Due to time constraints, all results used in this tutorial have been precomputed. Therefore, the corresponding data are retrieved from the [**NOMAD**](https://nomad-lab.eu/prod/v1/gui/search/entries) database. Otherwise, `exciting` would need to be installed and compiled, and the appropriate environment would need to be configured. More information about exciting and its usage can be found on [exciting webpage](https://www.exciting-code.org/).

<hr style="border:2px solid #DDD"> </hr>

## Introduction and physical background

Phonons are the quantized eigenmodes of the vibrations of the nuclei constituting the lattice in a crystalline solid. Mathematically, they are described by the eigenvalues and eigenvectors of the **dynamical matrix** 
$$ D_{\kappa\alpha,\kappa'\alpha'}({\bf q}) = - \left(M_\kappa\, M_{\kappa'}\right)^{-\frac{1}{2}}\, \sum_{\bf R} {\rm e}^{{\rm i}{\bf q}\cdot{\bf R}}\, \frac{\partial F_{\kappa\alpha}}{\partial \tau_{\kappa'{\bf R}\alpha'}} \;, $$
where $F_{\kappa\alpha}$ is the component in Cartesian direction $\alpha$ of the force acting on atom $\kappa$ with mass $M_\kappa$. The negative derivative of that force upon a displacement of atom $\kappa'$ in the unit cell at lattice vector ${\bf R}$ into direction ${\alpha'}$ is called the **interatomic force constant** (IFC) and the dynamical matrix is defined as the Fourier transform of the IFCs where ${\bf q}$ describes the wave vector or the wave length of corresponding vibration.

We calculate phonons using **density-functional perturbation theory** (DFPT), a method that aims at solving the first order derivative of the Kohn-Sham equations for the linear response of the electron density, the effective potential and the wave functions. Those first order derivatives can then be used to compute the derivative of the forces and thus the dynamical matrix.
Once the dynamical matrices have been obtained for given set of wave vectors ${\bf q}$, the corresponding **vibrational frequencies** $\omega_\nu({\bf q})$ and **atomic displacement patterns** $e_{\kappa\alpha,\nu}({\bf q})$ can obtained as
$$ \sum_{\kappa',\alpha'} D_{\kappa\alpha,\kappa\alpha'}({\bf q})\, e_{\kappa'\alpha',\nu}({\bf q}) = \omega^2_\nu({\bf q})\, e_{\kappa\alpha,\nu}({\bf q}) \;, $$
where $\nu$ labels the $3N_{\rm at}$ eigenpairs and is called the phonon branch.

In **polar materials** there is an additional contribution to the IFCs (and thus the dynamical matrices) coming from long ranging dipoles induced by the displacement of the incompletely screened ions. They can be described analytically in terms of the **dielectric constant** $\epsilon^\infty_{\alpha\alpha'}$ describing the anisotropic screening and the **Born effective charge tensors** $Z^\ast_{\alpha\alpha',\kappa}$ describing an effective charge of atom $\kappa$. The inclusion of that contribution to the dynamical matrices allows for the correct description of LO-TO splitting at the Brillouin zone center.

In this tutorial we will compute the dynamical matrices and the phonon dispersion $\omega_\nu({\bf q})$ and the phonon density of states (DOS)
$$ D(\omega) = \sum_\nu \int\limits_{\rm BZ}{\rm d}{\bf q}\, \delta(\omega - \omega_\nu({\bf q})) \;. $$

## Setup and usage of exciting

In this tutorial, we demonstrate how to run an exciting ground state calculation on the example of **$\beta$ - Ga<sub>2</sub>O<sub>3</sub>**.
In order to run an exciting calculation, we first need to generate an input.xml file which contains all the necessary input parameters. For this, we use the exciting python interface **excitingtools**. Details on this tool can be found in the [**excitingtools tutorial**](https://www.exciting-code.org/uploads/exciting/tutorial_notebooks/tutorial_excitingtools.html).

In [ ]:
from excitingtools.input import (ExcitingStructure, ExcitingInputXML, ExcitingGroundStateInput)
from excitingtools.input.input_classes import ExcitingLibxcInput

### Structure and ground state
First, we specify the structure and set up the ground state DFT calculation. For phonon calculations using DFPT, we recommend to specify the type of xc-functional using the `libxc` element of the ground state.

In [ ]:
species_path = "."

lattice = [[2.882457027, 11.61489328, 0.000000000],
           [-2.882457027, 11.61489328, 0.000000000],
           [0.000000000, 2.600503369, 10.67028355]]

atoms = [{'species': 'Ga', 'position': [0.0000000000, 0.0000000000, 0.0000000000]},
         {'species': 'Ga', 'position': [0.6822620800, 0.6822620800, 0.6280570400]},
         {'species': 'Ga', 'position': [0.2511389000, 0.2511389000, 0.1090277600]},
         {'species': 'Ga', 'position': [0.4311231800, 0.4311231800, 0.5190292800]},
         {'species': 'O', 'position': [0.1679453600, 0.1679453600, 0.8772238900]},
         {'species': 'O', 'position': [0.5143167200, 0.5143167200, 0.7508331500]},
         {'species': 'O', 'position': [0.1760095100, 0.1760095100, 0.4228546600]},
         {'species': 'O', 'position': [0.5062525700, 0.5062525700, 0.2052023800]},
         {'species': 'O', 'position': [0.8449338000, 0.8449338000, 0.5701798300]},
         {'species': 'O', 'position': [0.8373282800, 0.8373282800, 0.0578772100]}]

structure = ExcitingStructure(
  atoms, lattice, species_path, 
  autormt=False,
  species_properties={
    'Ga': {'rmt': 1.75}, 
    'O': {'rmt': 1.5}})

In [ ]:
groundstate = ExcitingGroundStateInput(
  do='fromscratch', 
  ngridk=[8,8,4], 
  gmaxvr=25.0,
  lmaxvr=12,
  lmaxapw=12,
  lmaxmat=12,
  rgkmax=8,
  outputlevel='high',
  maxscl=50,
  nempty=10)
groundstate.libxc = ExcitingLibxcInput(exchange='XC_GGA_X_PBE', correlation='XC_GGA_C_PBE')

exciting_input = ExcitingInputXML(
  structure=structure,
  groundstate=groundstate,
  title="Ga2O3 phonon calculation")

In [ ]:
from pathlib import Path
phonons_dir = Path("phonons_tutorial")
phonons_dir.mkdir(exist_ok=True)
input_xml = phonons_dir / "input.xml"
exciting_input.write(input_xml)

The main output of the **DFT** calculation for subsequent excited states calculations are stored in **STATE.OUT** and **EFERMI.OUT**. 

Since we use the mock runner and the binary file **STATE.OUT** is not transferable between maschines and compilers, we dont fetch **DFT** results seperately and directly proceed with the **Phonon** calculations. But we assume, we have the neccesary ground state calculation done, therefor we can skip the recomputation of the ground state by seting the following parameter in the input file. 

In [ ]:
exciting_input.groundstate.do = "skip"

### Phonons - dynamical matrices
Once the ground state calculation is completed we modify the input for the subsequent calculation of the dynamical matrices. We specify a grid of phonon wave vectors ${\bf q}$ for which to compute the dynamical matrices as well as the use of DFPT and the inclusion of the long-range dynamical matrix in polar materials.

In [ ]:
from excitingtools.input import ExcitingPhononsInput

phonons = ExcitingPhononsInput(
  do='fromscratch',
  ngridq=[4,4,4],
  method='dfpt',
  polar=True)

exciting_input.phonons = phonons

exciting_input.write(input_xml)

Now, we can run exciting using **excitingscripts**. This python library contains a lot of helpful scripts for running exciting and postprocessing exciting outputs. In this tutorial, we use the mock runner to fetch the pre-computed data from [**NOMAD**](https://nomad-lab.eu/prod/v1/gui/search/entries).

In [ ]:
%%bash
cd phonons_tutorial
python -m excitingscripts.dpg2026.mock_execute_single phonons --overwrite
cd ..

### Phonons - dispersion and DOS
Once the expensive calculation of the dynamical matrices is completed, we can additionally compute phonon properties such as the phonon dispersion or the phonon density of states. Both properties require the dynamical matrices for wave vectors ${\bf q}$ other than the ones on the grid that we have specified in the `phonons` element. This is conveniently achieved using Fourier interpolation to interpolate the dynamical matrices from the original grid to any point in reciprocal space. In order to converge the BZ integral appearing the DOS, we specify a dense integration grid on which the Fourier interpolation is to be applied.

In [ ]:
from excitingtools.input.input_classes import ExcitingPhonondispplotInput, ExcitingPhonondosInput

dos = ExcitingPhonondosInput(
  ngridqint=[20, 20, 20],
  nwdos=1000)

points = [
  {'coord': [0.00000000, 0.00000000, 0.00000000], 'label':'GAMMA'},
  {'coord': [0.50000000, 0.50000000, 0.00000000], 'label':'Y'},
  {'coord': [0.60315808, 0.60315808, 0.41000209], 'label':'F'},
  {'coord': [0.50000000, 0.50000000, 0.50000000], 'label':'L'},
  {'coord': [0.74187655, 0.25812345, 0.50000000], 'label':'I', 'breakafter':True},
  {'coord': [0.25812345, -0.25812345, 0.50000000], 'label':'I1'},
  {'coord': [0.00000000, 0.00000000, 0.00000000], 'label':'Z'},
  {'coord': [0.39684192, 0.39684192, 0.58999791], 'label':'F1', 'breakafter':True},
  {'coord': [0.50000000, 0.50000000, 0.00000000], 'label':'Y'},
  {'coord': [0.73364820, 0.26635180, 0.00000000], 'label':'X1', 'breakafter':True},
  {'coord': [0.26635180, -0.26635180, 0.00000000], 'label':'X'},
  {'coord': [0.00000000, 0.00000000, 0.00000000], 'label':'GAMMA'},
  {'coord': [0.50000000, 0.00000000, 0.00000000], 'label':'N', 'breakafter':True},
  {'coord': [0.50000000, 0.00000000, 0.50000000], 'label':'M'},
  {'coord': [0.00000000, 0.00000000, 0.00000000], 'label':'GAMMA'}]
dispersion = ExcitingPhonondispplotInput(plot1d={'path':{'steps': 480, 'point': points}})

phonons.do = 'skip'
phonons.phonondos = dos
phonons.phonondispplot = dispersion

exciting_input.write(input_xml)

**<span style="color:firebrick">Note</span>**: To avoid overhead, we already fetched the band-structure and DOS ouput files before. So we don't run exciting here. 

The generated DOS and band structure can be plotted using **excitingscripts**. We start with the DOS.

In [ ]:
%%bash
cd phonons_tutorial/DFPT-Ga2O3
rm PDOS.png
python -m excitingscripts.plot.dos -p -e 0 100 -fu meV  
mv PLOT.png PDOS.png
cd ../..

In [ ]:
from IPython.display import Image
Image(filename='phonons_tutorial/DFPT-Ga2O3/PDOS.png') 

Now, we can go on plotting the band structure.

In [ ]:
%%bash
cd phonons_tutorial/DFPT-Ga2O3
python -m excitingscripts.plot.band_structure -p -e 0 100 -fu meV -s 1.7 1
mv PLOT.png PDISP.png
cd ../..

In [ ]:
Image(filename='phonons_tutorial/DFPT-Ga2O3/PDISP.png') 

<hr style="border:2px solid #DDD"> </hr>
This concludes our tutorial on the computation of phonons using DFPT. You are encouraged to checkout the other methods implemented in exciting via the tutorials in this suite.
<hr style="border:2px solid #DDD"> </hr>